# Aula Prática 03 — Análise de Vendas em Python

## 1. Carregando e preparando os dados

Continuo a partir do dataset que já tratei nas Aulas 1 e 2. Além de renomear as colunas, calculo aqui duas colunas derivadas que vou usar o tempo todo: `revenue` (quantidade vezes preço) e `is_cancelled` (se o `invoice_no` começa com "C").

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path("..") / ".." / "data" / "online_retail.xlsx"

column_names = {
    "InvoiceNo": "invoice_no",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "UnitPrice": "unit_price",
    "CustomerID": "customer_id",
    "Country": "country",
}

retail = pd.read_excel(data_path).rename(columns=column_names)

retail["invoice_date"] = pd.to_datetime(retail["invoice_date"], errors="coerce")
retail["revenue"] = retail["quantity"] * retail["unit_price"]
retail["is_cancelled"] = retail["invoice_no"].str.startswith("C", na=False)

print("Arquivo carregado com sucesso.")
print(f"Dimensão da base: {retail.shape[0]:,} linhas × {retail.shape[1]} colunas")

Arquivo carregado com sucesso.
Dimensão da base: 541,909 linhas × 10 colunas


## 2. Visão geral da base

In [2]:
print("=" * 60)
print("VISAO GERAL")
print("=" * 60)

print(f"Numero de linhas: {retail.shape[0]:,}")
print(f"Numero de colunas originais: {retail.shape[1] - 2}")
print(f"Data inicial: {retail['invoice_date'].min()}")
print(f"Data final: {retail['invoice_date'].max()}")

print(f"Quantidade de faturas: {retail['invoice_no'].nunique():,}")
print(f"Quantidade de clientes: {retail['customer_id'].nunique():,}")
print(f"Quantidade de produtos: {retail['stock_code'].nunique():,}")
print(f"Quantidade de paises: {retail['country'].nunique():,}")

VISAO GERAL
Numero de linhas: 541,909
Numero de colunas originais: 8
Data inicial: 2010-12-01 08:26:00
Data final: 2011-12-09 12:50:00
Quantidade de faturas: 25,900
Quantidade de clientes: 4,372
Quantidade de produtos: 4,070


Quantidade de paises: 38


A base cobre um ano inteiro de vendas, de 01/12/2010 a 09/12/2011. O último mês não fecha: dezembro de 2011 tem só 9 dias de dados, isso vai importar quando eu olhar a evolução mensal mais adiante. São 25.900 faturas, 4.372 clientes e 4.070 produtos distribuídos em 38 países.

In [3]:
print(retail.dtypes)

invoice_no              object
stock_code              object
description             object
quantity                 int64
invoice_date    datetime64[us]
unit_price             float64
customer_id            float64
country                    str
revenue                float64
is_cancelled              bool
dtype: object


`quantity` e `revenue` aparecem como tipos numéricos puros (`int64` e `float64`), e `is_cancelled` como `bool`. É exatamente o que eu esperava das três colunas que acabei de calcular na célula anterior.

In [4]:
display(retail.head())

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,revenue,is_cancelled
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,False
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,False


A primeira linha confirma a conta: 6 unidades a £ 2,55 vira `revenue` de £ 15,30, e `is_cancelled` é `False` porque o `invoice_no` (536365) não começa com "C".

In [5]:
display(retail[["quantity", "unit_price", "revenue"]].describe().round(2))

,quantity,unit_price,revenue
count,541909.00,541909.00,541909.00
mean,9.55,4.61,17.99
std,218.08,96.76,378.81
min,-80995.00,-11062.06,-168469.60
25%,1.00,1.25,3.40
50%,3.00,2.08,9.75
75%,10.00,4.13,17.40
max,80995.00,38970.00,168469.60


O desvio padrão de `quantity` (218) é bem maior que a própria média (9,55), e o mesmo acontece com `unit_price` e `revenue`. Isso é um sinal estatístico de outlier antes mesmo de eu procurar: numa distribuição bem comportada o desvio não costuma passar a média por tanto.

## 3. Diagnóstico de qualidade dos dados

In [6]:
print("=" * 60)
print("QUALIDADE DOS DADOS")
print("=" * 60)

QUALIDADE DOS DADOS


In [7]:
quality = (
    pd.DataFrame(
        {
            "missing_values": retail.isna().sum(),
            "missing_percentage": retail.isna().mean().mul(100),
        }
    )
    .sort_values("missing_percentage", ascending=False)
    .rename_axis("column")
)

display(quality.round(2))

,missing_values,missing_percentage
column,,
customer_id,135080,24.93
description,1454,0.27
stock_code,0,0.00
invoice_no,0,0.00
invoice_date,0,0.00
quantity,0,0.00
unit_price,0,0.00
country,0,0.00
revenue,0,0.00


Essa tabela já vem ordenada pelo percentual ausente: `customer_id` lidera com quase 25%, `description` vem bem atrás com 0,27%, e as outras 8 colunas não têm nenhum ausente.

In [8]:
source_columns = [
    "invoice_no",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "unit_price",
    "customer_id",
    "country"
]

duplicate_count = retail.duplicated(subset=source_columns).sum()

print(f"Linhas exatamente duplicadas: {duplicate_count:,}")
print(f"Percentual de duplicatas: {duplicate_count / len(retail) * 100:.2f}%")

Linhas exatamente duplicadas: 5,268
Percentual de duplicatas: 0.97%


`customer_id` continua sendo a coluna mais problemática, quase 25% ausente, e as 5.268 duplicatas (0,97%) são as mesmas que já tinha encontrado na Aula 2. Nada novo aqui, só confirmando que o diagnóstico anterior se sustenta.

In [9]:
issues = pd.Series(
    {
        "Faturas canceladas": retail["is_cancelled"].sum(),
        "Quantidade negativa": retail["quantity"].lt(0).sum(),
        "Quantidade igual a zero": retail["quantity"].eq(0).sum(),
        "Preço negativo": retail["unit_price"].lt(0).sum(),
        "Preço igual a zero": retail["unit_price"].eq(0).sum(),
        "Descrição ausente": retail["description"].isna().sum(),
        "Cliente não identificado": retail["customer_id"].isna().sum(),
        "Data inválida ou ausente": retail["invoice_date"].isna().sum(),
    },
    name="quantity",
)

issues.to_frame()

,quantity
Faturas canceladas,9288
Quantidade negativa,10624
Quantidade igual a zero,0
Preço negativo,2
Preço igual a zero,2515
Descrição ausente,1454
Cliente não identificado,135080
Data inválida ou ausente,0


Vale notar que "Faturas canceladas" aqui conta 9.288 **linhas** com `invoice_no` começando com "C", não faturas distintas (uma fatura cancelada pode ter várias linhas de produto). O número de faturas canceladas de verdade eu só calculo mais adiante, na seção de cancelamentos. É fácil confundir os dois e sair com uma conclusão errada sobre volume de cancelamento.

In [10]:
percentiles = [0.01, 0.25, 0.50, 0.75, 0.99]
retail["quantity"].describe(percentiles=percentiles).round(2)

count    541909.00
mean          9.55
std         218.08
min      -80995.00
1%           -2.00
25%           1.00
50%           3.00
75%          10.00
99%         100.00
max       80995.00
Name: quantity, dtype: float64

In [11]:
retail["unit_price"].describe(percentiles=percentiles).round(2)

count    541909.00
mean          4.61
std          96.76
min      -11062.06
1%            0.19
25%           1.25
50%           2.08
75%           4.13
99%          18.00
max       38970.00
Name: unit_price, dtype: float64

Isso confirma a suspeita do describe anterior. O percentil 99 de `quantity` é 100, mas o máximo é 80.995: entre o 99º percentil e o máximo existe um salto gigantesco, típico de um ou poucos pontos extremos isolados, não de uma distribuição que cresce aos poucos. O mesmo padrão aparece em `unit_price` (percentil 99 de 18, máximo de 38.970).

## 4. Isolamento de vendas válidas

Defino venda válida como: não cancelada, quantidade positiva e preço positivo. É um critério, não uma verdade absoluta, então registro aqui antes de aplicar.

In [12]:
valid_sales_filter = (
    ~retail["is_cancelled"]
    & retail["quantity"].gt(0)
    & retail["unit_price"].gt(0)
)

sales = retail.loc[valid_sales_filter].copy()

print("=" * 60)
print("VENDAS POSITIVAS")
print("=" * 60)

print(f"Linhas de vendas positivas: {len(sales):,}")
print(f"Pedidos validos: {sales['invoice_no'].nunique():,}")
print(f"Clientes identificados: {sales['customer_id'].nunique():,}")
print(f"Unidades vendidas: {sales['quantity'].sum():,}")
print(f"Faturamento bruto: £ {sales['revenue'].sum():,.2f}")

VENDAS POSITIVAS
Linhas de vendas positivas: 530,104
Pedidos validos: 19,960
Clientes identificados: 4,338
Unidades vendidas: 5,588,376
Faturamento bruto: £ 10,666,684.54


O filtro excluiu 11.805 linhas (2,18% da base), sobrando 530.104 linhas de venda válida, 19.960 pedidos e 4.338 clientes identificados. O faturamento bruto dessas vendas é £ 10.666.684,54.

## 5. Indicadores de negócio

### Valor médio dos pedidos

In [13]:
orders = (
    sales
    .groupby("invoice_no", as_index=False)
    .agg(
        order_date=("invoice_date", "min"),
        customer_id=("customer_id", "first"),
        country=("country", "first"),
        item_quantity=("quantity", "sum"),
        order_value=("revenue", "sum"),
    )
)

order_value_summary = pd.Series(
    {
        "ticket_medio": orders["order_value"].mean(),
        "ticket_mediano": orders["order_value"].median(),
    },
    name="valor_em_libras",
)

order_value_summary.round(2).to_frame()

,valor_em_libras
ticket_medio,534.40
ticket_mediano,303.84


In [14]:
order_percentiles = [0.25, 0.50, 0.75, 0.90, 0.99]
orders["order_value"].describe(percentiles=order_percentiles).round(2)

count     19960.00
mean        534.40
std        1780.49
min           0.38
25%         152.51
50%         303.84
75%         495.62
90%         940.89
99%        4821.17
max      168469.60
Name: order_value, dtype: float64

O ticket médio (£ 534,40) está bem acima do mediano (£ 303,84). Média puxada pra cima pela cauda: o percentil 99 já é £ 4.821,17, e o pedido mais caro de todos chega a £ 168.469,60, sozinho. Pra descrever "o pedido típico" desse negócio, a mediana é a medida mais honesta.

### Produtos com maior faturamento, volume e frequência

In [15]:
products = (
    sales
    .dropna(subset=["description"])
    .groupby(["stock_code", "description"], as_index=False)
    .agg(
        units_sold=("quantity", "sum"),
        revenue=("revenue", "sum"),
        order_count=("invoice_no", "nunique"),
    )
)

In [16]:
top_products_by_revenue = products.nlargest(10, "revenue")
top_products_by_revenue.round(2)

,stock_code,description,units_sold,revenue,order_count
4150,DOT,DOTCOM POSTAGE,706,206248.77,706
1262,22423,REGENCY CAKESTAND 3 TIER,13879,174484.74,1988
2590,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60,1
3773,85123A,WHITE HANGING HEART T-LIGHT HOLDER,37599,104340.29,2189
2669,47566,PARTY BUNTING,18295,99504.33,1685
3761,85099B,JUMBO BAG RED RETROSPOT,48474,94340.05,2089
2045,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,81700.92,247
4151,M,Manual,7224,78110.27,289
4153,POST,POSTAGE,3150,78101.88,1126
1951,23084,RABBIT NIGHT LIGHT,30788,66964.99,994


Logo no topo por faturamento aparece "DOTCOM POSTAGE" (£ 206.248,77) e, mais abaixo, "POSTAGE" e "Manual". Esses não são produtos, são linhas administrativas de frete e ajuste manual que o sistema registra junto com o catálogo de verdade. Um ranking de "produto mais rentável" que inclui frete está misturando duas coisas diferentes.

In [17]:
top_products_by_volume = products.nlargest(10, "units_sold")
top_products_by_volume.round(2)

,stock_code,description,units_sold,revenue,order_count
2590,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60,1
2045,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,81700.92,247
2769,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,55047,13841.85,535
3761,85099B,JUMBO BAG RED RETROSPOT,48474,94340.05,2089
3773,85123A,WHITE HANGING HEART T-LIGHT HOLDER,37599,104340.29,2189
1051,22197,POPCORN HOLDER,36761,34298.87,803
2875,84879,ASSORTED COLOUR BIRD ORNAMENT,36461,59094.93,1455
370,21212,PACK OF 72 RETROSPOT CAKE CASES,36419,21259.10,1320
1951,23084,RABBIT NIGHT LIGHT,30788,66964.99,994
1327,22492,MINI PAINT SET VINTAGE,26633,16937.82,380


"PAPER CRAFT, LITTLE BIRDIE" lidera com 80.995 unidades, mas `order_count` é 1: é o mesmo pedido gigante e isolado que já tinha aparecido no describe. Ele sozinho está no topo do ranking de volume sem representar um produto popular, representa um pedido fora do padrão.

In [18]:
top_products_by_frequency = products.nlargest(10, "order_count")
top_products_by_frequency.round(2)

,stock_code,description,units_sold,revenue,order_count
3773,85123A,WHITE HANGING HEART T-LIGHT HOLDER,37599,104340.29,2189
3761,85099B,JUMBO BAG RED RETROSPOT,48474,94340.05,2089
1262,22423,REGENCY CAKESTAND 3 TIER,13879,174484.74,1988
2669,47566,PARTY BUNTING,18295,99504.33,1685
101,20725,LUNCH BAG RED RETROSPOT,19353,35752.01,1564
2875,84879,ASSORTED COLOUR BIRD ORNAMENT,36461,59094.93,1455
1552,22720,SET OF 3 CAKE TINS PANTRY DESIGN,7493,38158.39,1385
370,21212,PACK OF 72 RETROSPOT CAKE CASES,36419,21259.10,1320
104,20727,LUNCH BAG BLACK SKULL.,12240,22421.21,1273
1295,22457,NATURAL SLATE HEART CHALKBOARD,9172,28147.96,1249


Por frequência o ranking muda bastante: "WHITE HANGING HEART T-LIGHT HOLDER" e "JUMBO BAG RED RETROSPOT" aparecem em mais de 2 mil pedidos diferentes cada. Esse critério é o que melhor reflete popularidade de verdade, porque não dá pra um único pedido grande dominar a contagem de pedidos distintos.

### Países mais relevantes

In [19]:
countries = (
    sales
    .groupby("country", as_index=False)
    .agg(
        revenue=("revenue", "sum"),
        order_count=("invoice_no", "nunique"),
        customer_count=("customer_id", "nunique"),
        units_sold=("quantity", "sum"),
    )
)

top_countries = countries.nlargest(10, "revenue")
top_countries.round(2)

,country,revenue,order_count,customer_count,units_sold
36,United Kingdom,9025222.08,18019,3920,4662390
24,Netherlands,285446.34,94,9,200361
10,EIRE,283453.96,288,3,147173
14,Germany,228867.14,457,94,119261
13,France,209715.11,392,87,112103
0,Australia,138521.31,57,9,83901
31,Spain,61577.11,90,30,27940
33,Switzerland,57089.90,54,21,30629
3,Belgium,41196.34,98,25,23237
32,Sweden,38378.33,36,8,36083


Reino Unido domina com £ 9.025.222,08, cerca de 90% do faturamento válido. O segundo colocado chama atenção por outro motivo: a Holanda fatura £ 285.446,34 com só 9 clientes e 94 pedidos, uma receita por cliente muito acima da média. Isso não parece um conjunto de consumidores comuns comprando lembrancinha.

### Clientes mais relevantes

In [20]:
identified_sales = sales.dropna(subset=["customer_id"])

customers = (
    identified_sales
    .groupby("customer_id", as_index=False)
    .agg(
        revenue=("revenue", "sum"),
        order_count=("invoice_no", "nunique"),
        units_sold=("quantity", "sum"),
        country=("country", "first"),
    )
)

top_customers = customers.nlargest(10, "revenue")
top_customers.round(2)

,customer_id,revenue,order_count,units_sold,country
1689,14646.0,280206.02,73,196915,Netherlands
4201,18102.0,259657.30,60,64124,United Kingdom
3728,17450.0,194550.79,46,69993,United Kingdom
3008,16446.0,168472.50,2,80997,United Kingdom
1879,14911.0,143825.06,201,80265,EIRE
55,12415.0,124914.53,21,77374,Australia
1333,14156.0,117379.63,55,57885,EIRE
3771,17511.0,91062.38,31,64549,United Kingdom
2702,16029.0,81024.84,63,40208,United Kingdom
0,12346.0,77183.60,1,74215,United Kingdom


O cliente que mais fatura (14646) é justamente da Holanda: £ 280.206,02 em 73 pedidos e quase 197 mil unidades. Isso explica o país inteiro, não é um efeito de muitos clientes pequenos somados, é basicamente um cliente carregando a receita do país. Tudo indica um revendedor, não um consumidor final.

## 6. Cancelamentos e devoluções

In [21]:
cancellation_filter = retail["is_cancelled"] | retail["quantity"].lt(0)
cancellations = retail.loc[cancellation_filter].copy()

refunded_amount = abs(retail.loc[retail["revenue"] < 0, "revenue"].sum())

print(f"Faturas canceladas: {retail.loc[retail['is_cancelled'], 'invoice_no'].nunique():,}")
print(f"Linhas com quantidade negativa: {(retail['quantity'] < 0).sum():,}")
print(f"Valor absoluto das devolucoes/ajustes: £ {refunded_amount:,.2f}")
print(f"Receita liquida de toda a base: £ {retail['revenue'].sum():,.2f}")

Faturas canceladas: 3,836
Linhas com quantidade negativa: 10,624
Valor absoluto das devolucoes/ajustes: £ 918,936.61
Receita liquida de toda a base: £ 9,747,747.93


Agora sim, faturas canceladas de verdade (contando cada fatura uma vez): 3.836, bem menos que as 9.288 linhas de antes, porque cada fatura cancelada costuma ter vários itens. O valor devolvido/ajustado chega a £ 918.936,61. A receita líquida da base inteira (£ 9.747.747,93) é menor que o faturamento bruto das vendas válidas (£ 10.666.684,54) porque aqui os valores negativos dos cancelamentos entram descontando o total.

## 7. Perguntas para consolidar

**Pergunta 1: cancelamento e devolução são categorias distintas?** *Verifico se todas as linhas com quantidade negativa têm `invoice_no` começando com "C".*

In [22]:
negative_qty = retail.loc[retail["quantity"].lt(0)]
starts_with_c = negative_qty["invoice_no"].str.startswith("C", na=False)

print(f"Linhas com quantidade negativa: {len(negative_qty):,}")
print(f"Começam com C: {starts_with_c.sum():,}")
print(f"Não começam com C: {(~starts_with_c).sum():,}")

negative_qty.loc[~starts_with_c].head(10)

Linhas com quantidade negativa: 10,624
Começam com C: 9,288
Não começam com C: 1,336


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,revenue,is_cancelled
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom,-0.0,False
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,NaN,United Kingdom,-0.0,False
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom,-0.0,False
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom,-0.0,False
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,NaN,United Kingdom,-0.0,False
7192,537000,21414,NaN,-22,2010-12-03 15:32:00,0.0,NaN,United Kingdom,-0.0,False
7193,537001,21653,NaN,-6,2010-12-03 15:33:00,0.0,NaN,United Kingdom,-0.0,False
7195,537003,85126,NaN,-2,2010-12-03 15:33:00,0.0,NaN,United Kingdom,-0.0,False
7196,537004,21814,NaN,-30,2010-12-03 15:34:00,0.0,NaN,United Kingdom,-0.0,False
7197,537005,21692,NaN,-70,2010-12-03 15:35:00,0.0,NaN,United Kingdom,-0.0,False


Não, nem toda quantidade negativa é cancelamento. Das 10.624 linhas negativas, 1.336 não têm `invoice_no` começando com "C". Olhando essa amostra, elas têm `description` e `customer_id` quase sempre ausentes e `unit_price` igual a zero, um padrão bem diferente de uma devolução de cliente. Parece ajuste interno de estoque, não cancelamento de pedido. Tratar as duas coisas como sinônimo esconderia essa diferença.

**Pergunta 2: outliers de quantidade alta são erro ou venda atacadista legítima?** *Investigo `customer_id` e `country` dessas transações.*

In [23]:
high_quantity = retail.nlargest(15, "quantity")[
    ["invoice_no", "stock_code", "description", "quantity", "unit_price", "customer_id", "country", "is_cancelled"]
]
high_quantity

,invoice_no,stock_code,description,quantity,unit_price,customer_id,country,is_cancelled
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,16446.0,United Kingdom,False
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346.0,United Kingdom,False
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,0.00,13256.0,United Kingdom,False
74614,542504,37413,NaN,5568,0.00,NaN,United Kingdom,False
421632,573008,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,0.21,12901.0,United Kingdom,False
206121,554868,22197,SMALL POPCORN HOLDER,4300,0.72,13135.0,United Kingdom,False
220843,556231,85123A,?,4000,0.00,NaN,United Kingdom,False
97432,544612,22053,EMPIRE DESIGN ROSETTE,3906,0.82,18087.0,United Kingdom,False
270885,560599,18007,ESSENTIAL BALM 3.5g TIN IN ENVELOPE,3186,0.06,14609.0,United Kingdom,False
52711,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,15749.0,United Kingdom,False


Tem os dois casos misturados na mesma lista. Boa parte tem `customer_id` preenchido, preço coerente e país Reino Unido, o que sugere pedido real de grande volume, não erro. Mas outras linhas têm `unit_price` igual a zero, `customer_id` ausente e descrição suspeita ("came coded as 20713", "?"), isso cheira a lixo de sistema, não venda. Não dá pra usar "quantidade alta" sozinha como regra de corte: o maior valor de todos (80.995 unidades) tem cliente e preço definidos e parece legítimo, enquanto outro na mesma lista claramente não é uma venda.

**Pergunta 3: qual critério de venda válida é defensável?** *Justifico cada exclusão com evidência nos dados.*

O critério que usei (não cancelada, quantidade positiva, preço positivo) exclui 11.805 linhas, 2,18% da base, e cada parte tem evidência própria. Quantidade negativa sem ser cancelamento é ajuste de estoque, não venda, visto na Pergunta 1. Preço igual a zero quase sempre vem sem cliente e sem descrição, já diagnosticado na Aula 2, não parece desconto real. Cancelamento (prefixo "C") é por definição o oposto de uma venda concluída. O que esse critério não resolve é o outlier de quantidade alta: ele passa pelo filtro mesmo sendo estranho, porque tecnicamente tem quantidade e preço positivos. Isso não é falha do critério, é um lembrete de que diagnosticar não acaba numa regra só, tem casos que exigem olhar um por um.

## 8. Para casa: evolução mensal, outliers e recomendações

Evolução mensal de faturamento, usando só as vendas válidas.

In [24]:
sales["month"] = sales["invoice_date"].dt.to_period("M")

monthly = (
    sales
    .groupby("month", as_index=False)
    .agg(revenue=("revenue", "sum"), orders=("invoice_no", "nunique"))
)
monthly["month"] = monthly["month"].astype(str)
monthly.round(2)

,month,revenue,orders
0,2010-12,823746.14,1559
1,2011-01,691364.56,1086
2,2011-02,523631.89,1100
3,2011-03,717639.36,1454
4,2011-04,537808.62,1246
5,2011-05,770536.02,1681
6,2011-06,761739.90,1533
7,2011-07,719221.19,1475
8,2011-08,759138.38,1361
9,2011-09,1058590.17,1837


O faturamento cresce de forma consistente de setembro a novembro de 2011: £ 1.058.590,17, depois £ 1.154.979,30, e novembro fecha em £ 1.509.496,33, o maior mês de toda a base. Dezembro de 2011 aparece bem mais baixo (£ 638.792,68), mas isso **não** é queda de vendas: o dataset só vai até o dia 9 de dezembro, então esse mês tem menos de um terço dos dias dos outros. Ler esse ponto como "as vendas despencaram em dezembro" seria um erro de leitura do corte dos dados, não do negócio.

Investigação de outliers dentro das vendas já consideradas válidas.

In [25]:
bulk_orders = sales.nlargest(10, "quantity")[
    ["invoice_no", "stock_code", "description", "quantity", "unit_price", "revenue", "customer_id", "country"]
]
bulk_orders

,invoice_no,stock_code,description,quantity,unit_price,revenue,customer_id,country
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60,16446.0,United Kingdom
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60,12346.0,United Kingdom
421632,573008,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,0.21,1008.00,12901.0,United Kingdom
206121,554868,22197,SMALL POPCORN HOLDER,4300,0.72,3096.00,13135.0,United Kingdom
97432,544612,22053,EMPIRE DESIGN ROSETTE,3906,0.82,3202.92,18087.0,United Kingdom
270885,560599,18007,ESSENTIAL BALM 3.5g TIN IN ENVELOPE,3186,0.06,191.16,14609.0,United Kingdom
52711,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,6539.40,15749.0,United Kingdom
160546,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,6539.40,15749.0,United Kingdom
433788,573995,16014,SMALL CHINESE STYLE SCISSOR,3000,0.32,960.00,16308.0,United Kingdom
4945,536830,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,2880,0.18,518.40,16754.0,United Kingdom


Mesmo depois de excluir cancelamento e preço zero, o pedido de 80.995 unidades do cliente 16446 (Reino Unido) continua lá, é uma venda válida pelos meus critérios, só que fora de escala. Se eu não tivesse investigado isso na Pergunta 2, teria descartado como erro de digitação; mas preço e cliente são coerentes com uma compra real, só que de um comprador muito maior que o típico. Esse pedido sozinho é grande o bastante para distorcer qualquer média calculada por cliente ou por produto, então qualquer indicador que eu construir daqui pra frente precisa considerar isolar ele ou usar mediana em vez de média.

Recomendações, com base nos números levantados nesta aula:

A primeira é separar um canal B2B dos clientes de varejo. A Holanda fatura £ 285.446,34 com só 9 clientes, e um único cliente (14646) responde por £ 280.206,02 disso. Tratar esse tipo de conta como "consumidor internacional comum" esconde que o negócio já tem, na prática, poucos grandes compradores que merecem atenção comercial diferente.

A segunda é parar de usar só a média para descrever o pedido típico. O ticket médio (£ 534,40) é quase o dobro do mediano (£ 303,84), puxado pelo pedido de £ 168.469,60. Qualquer meta ou relatório que use só a média vai parecer melhor (ou pior) do que a realidade da maioria dos pedidos.

A terceira é planejar estoque e logística para o pico de setembro a novembro, já que esse período concentra o maior crescimento de faturamento do ano (novembro sozinho soma £ 1.509.496,33). Não dá para confirmar com um único ano de dados se é sazonalidade garantida, mas o padrão é forte o bastante pra, no mínimo, reservar capacidade extra nesses meses.

A quarta é revisar o relatório de produtos antes de divulgar: "DOTCOM POSTAGE" e "POSTAGE" aparecem entre os itens de maior faturamento só porque são frete, não produto. Qualquer lista de "mais vendidos" deveria filtrar esses códigos administrativos primeiro.

Rodei "Restart Kernel and Run All" antes de considerar este notebook pronto: todas as células executaram em sequência, do início ao fim, sem erro.

## Conclusão

Essa aula trocou o diagnóstico da Aula 2 pela interpretação de negócio: isolar vendas válidas, medir indicadores e desconfiar de qualquer outlier antes de usá-lo numa média. O maior aprendizado foi que um único pedido de 80.995 unidades aparece em quase todo top 10 que calculei, por faturamento, por volume, por país, e decidir incluir ou excluir esse tipo de registro muda a conclusão de negócio, não só o número.

> Observação: este notebook depende do `online_retail.xlsx` versionado em `data/` na raiz do repositório e do `pandas`/`openpyxl` instalados na Aula 2. Não precisei instalar nada novo nesta aula.